In [1]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, datetime, gc
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer
from pyspark.sql import functions as F
from pyspark.mllib.evaluation import RankingMetrics

# 1. Cấu hình đường dẫn
BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_FILE = BASE_PATH + "processed/cleaned_transactions.parquet"
OUTPUT_DIR = BASE_PATH + "outputs/"
CHECKPOINT_DIR = BASE_PATH + "spark_checkpoints/"

os.makedirs(OUTPUT_DIR + "candidates/", exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 2. Khởi tạo Spark (10GB RAM)
spark = SparkSession.builder \
    .appName("HM_ALS_ID_Standardization") \
    .config("spark.driver.memory", "10g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setCheckpointDir(CHECKPOINT_DIR)
print("✅ Spark Ready!")

Mounted at /content/drive
✅ Spark Ready!


In [2]:
# 1. Đọc dữ liệu
df = spark.read.parquet(INPUT_FILE)
max_date = df.select(F.max("t_dat_date")).collect()[0][0]

# 2. Các mốc thời gian
test_start_date = max_date - datetime.timedelta(days=7)       # Tuần 8
val_start_date = test_start_date - datetime.timedelta(days=7)  # Tuần 7
train_start_date = val_start_date - datetime.timedelta(weeks=6) # Tuần 1-6

# 3. Chia tập dữ liệu
train_retrieval = df.filter((F.col("t_dat_date") >= train_start_date) & (F.col("t_dat_date") < val_start_date))
val_ranking = df.filter((F.col("t_dat_date") >= val_start_date) & (F.col("t_dat_date") < test_start_date))

# 4. String Indexing (Chỉ fit trên tập Train W1-6)
u_model = StringIndexer(inputCol="customer_id", outputCol="user_idx").setHandleInvalid("skip").fit(train_retrieval)
i_model = StringIndexer(inputCol="article_id", outputCol="item_idx").setHandleInvalid("skip").fit(train_retrieval)

# Chuẩn bị đáp án Tuần 7 để đánh giá Recall/MAP nội bộ
val_gt_indexed = i_model.transform(u_model.transform(val_ranking)) \
    .groupBy("user_idx") \
    .agg(F.collect_list("item_idx").alias("actual_item_idxs"))

print(f"📅 Train (W1-6): {train_start_date} -> {val_start_date}")

📅 Train (W1-6): 2020-07-28 -> 2020-09-08


In [3]:
BEST_DECAY = 0.1
BEST_RANK = 80
BEST_ALPHA = 40

print("🚀 Đang huấn luyện ALS...")

# Tính Ratings có Time Decay
ratings = train_retrieval.withColumn("days_diff", F.datediff(F.lit(val_start_date), F.col("t_dat_date"))) \
                         .withColumn("weight", F.exp(-BEST_DECAY * F.col("days_diff"))) \
                         .groupBy("customer_id", "article_id") \
                         .agg(F.sum("weight").alias("rating"))

train_indexed = i_model.transform(u_model.transform(ratings)).checkpoint()

# Huấn luyện mô hình
als_model = ALS(maxIter=15, rank=BEST_RANK, regParam=0.1, alpha=BEST_ALPHA,
                userCol="user_idx", itemCol="item_idx", ratingCol="rating",
                implicitPrefs=True, coldStartStrategy="drop", nonnegative=True).fit(train_indexed)

print("✅ Huấn luyện ALS hoàn tất!")

🚀 Đang huấn luyện ALS...
✅ Huấn luyện ALS hoàn tất!


In [4]:
print("🎯 Đang tạo 100 ứng viên và đánh giá...")

# Tạo 100 ứng viên
val_recs = als_model.recommendForUserSubset(val_gt_indexed.select("user_idx"), 100).cache()

# Tính MAP@12 để kiểm tra sức mạnh mô hình
eval_rdd = val_recs.join(val_gt_indexed, "user_idx") \
    .select(F.col("recommendations.item_idx").alias("p"), "actual_item_idxs") \
    .rdd.map(lambda r: (list(r[0]), list(r[1])))

map12 = RankingMetrics(eval_rdd.map(lambda x: (x[0][:12], x[1]))).meanAveragePrecision
print(f"🏆 MAP@12 (Pure ALS trên Tuần 7): {map12:.6f}")

🎯 Đang tạo 100 ứng viên và đánh giá...


/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


🏆 MAP@12 (Pure ALS trên Tuần 7): 0.014845


In [5]:
print("💾 Đang giải mã Index sang ID gốc...")

# 1. Tạo bảng tra cứu từ Indexer
user_map = spark.createDataFrame([(i, s) for i, s in enumerate(u_model.labels)], ["user_idx", "customer_id"])
item_map = spark.createDataFrame([(i, s) for i, s in enumerate(i_model.labels)], ["item_idx", "article_id"])

# 2. Giải mã và chuẩn hóa mã 10 số cho Article_ID
# Chúng ta Explode để tạo tập dữ liệu "phẳng" cho XGBoost
val_recs_decoded = val_recs.select("user_idx", F.explode("recommendations").alias("rec")) \
    .select("user_idx", "rec.item_idx", F.col("rec.rating").alias("als_score")) \
    .join(user_map, "user_idx") \
    .join(item_map, "item_idx") \
    .withColumn("article_id", F.lpad(F.col("article_id").cast("string"), 10, "0")) \
    .select("customer_id", "article_id", "als_score")

# 3. Lưu file ứng viên (Bỏ phần lưu nhãn theo yêu cầu của Leader)
val_recs_decoded.write.mode("overwrite").parquet(OUTPUT_DIR + "candidates/als_top100_val_W7_decoded.parquet")

# 4. Lưu lại Model và Indexer để dự phòng
als_model.write().overwrite().save(OUTPUT_DIR + "models/als_best_model")
u_model.write().overwrite().save(OUTPUT_DIR + "models/user_indexer")
i_model.write().overwrite().save(OUTPUT_DIR + "models/item_indexer")

print(f"🏆 MISSION ACCOMPLISHED! File ứng viên ID gốc đã sẵn sàng tại: {OUTPUT_DIR}candidates/")

💾 Đang giải mã Index sang ID gốc...
🏆 MISSION ACCOMPLISHED! File ứng viên ID gốc đã sẵn sàng tại: /content/drive/MyDrive/HM-DATA/outputs/candidates/
